# Quick test: SFT v1 vs SFT v2

A small, fast comparison of the two SFT models on **the same unseen conversations** (about 15 minutes on a T4). It is a check, not the full evaluation: the evaluation notebook (Chapter 7) remains the real comparison.

| Test | What it tells you |
|---|---|
| Likelihood of the real reply (final reply only, same conversations, both models) | Which model predicts Ahmed's actual replies better. The training losses of v1 (1.625) and v2 cannot be compared; this can |
| Private names in generated replies (counted, never printed) | Whether v2 leaks names less than v1 |
| Reply length vs Ahmed's real replies | Whether a model talks too much |
| Three fixed prompts, side by side | A quick look with your own eyes |
| Chat with v2 | Try your own message |

The conversations come from the **redacted validation file**: never trained on by either model. The models are loaded **one at a time** (two 3B models do not fit on a T4). Only totals are printed and saved, unless you set `SHOW_EXAMPLES = True`.

**Before committing: Edit → Clear all outputs.**

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import torch
assert torch.cuda.is_available(), "No GPU -- Runtime > Change runtime type > GPU"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
!pip install unsloth

In [ ]:
import unsloth  # must come before transformers
from unsloth import FastLanguageModel

import gc
import json
import os
import random
import re
import statistics

import torch.nn.functional as F
from scipy.stats import binomtest

DRIVE_ROOT = "/content/drive/MyDrive"
PROJECT_DIR = f"{DRIVE_ROOT}/ahmed_twin"
V1_PATH = f"{PROJECT_DIR}/model_merged_nilechat"
V2_PATH = f"{PROJECT_DIR}/model_merged_nilechat_v2"
VAL_PATH = f"{PROJECT_DIR}/redacted/validation.jsonl"
PRIVATE_NAMES_PATH = f"{PROJECT_DIR}/private_names.txt"
RESULTS_PATH = f"{PROJECT_DIR}/sft_runs/sft_v2_redacted_final_reply_loss/quick_test_v1_vs_v2.json"

N_CONVERSATIONS = 60      # more = slower but more reliable
SEED = 0
MAX_SEQ_LENGTH = 2048
MAX_NEW_TOKENS = 100
SHOW_EXAMPLES = False     # True prints real conversations and replies: private, clear outputs afterwards
TEST_PROMPTS = ["احكيلي عن نفسك", "معلش هو انت فاضي دلوقتي؟", "عامل ايه النهاردة؟"]

for path in (V1_PATH, V2_PATH, VAL_PATH):
    assert os.path.exists(path), f"Missing {path}"

summary_path = os.path.join(V2_PATH, "sft_run_summary.json")
if os.path.exists(summary_path):
    with open(summary_path, encoding="utf-8") as f:
        v2_summary = json.load(f)
    print(f"v2 was trained to step {v2_summary['stopped_at_step']}, best eval loss {v2_summary['best_eval_loss']:.4f} "
          f"at {os.path.basename(str(v2_summary['best_checkpoint']))}")

## 2. Private names (same rules as `tools/private_names_lib.py`)

Whole words, with an attached و/ب/ل prefix. Used only to **count**.

In [ ]:
ARABIC = "؀-ۿ"
NAME_PREFIXES = "وبل"


def load_private_names(path):
    names = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#"):
                names.append(line)
    return sorted(set(names), key=len, reverse=True)


def compile_name_patterns(names):
    arabic = [n for n in names if re.search(f"[{ARABIC}]", n)]
    latin = [n for n in names if n not in arabic]
    ar = (re.compile(rf"(?<![{ARABIC}])(?P<pre>[{NAME_PREFIXES}])?(?P<name>{'|'.join(map(re.escape, arabic))})(?![{ARABIC}])")
          if arabic else None)
    la = re.compile(rf"(?<![A-Za-z])(?:{'|'.join(map(re.escape, latin))})(?![A-Za-z])", re.IGNORECASE) if latin else None
    return ar, la


def find_private_names(text, patterns):
    ar, la = patterns
    found = []
    if ar:
        found += [m.group("name") for m in ar.finditer(text)]
    if la:
        found += [m.group(0) for m in la.finditer(text)]
    return found


NAME_PATTERNS = compile_name_patterns(load_private_names(PRIVATE_NAMES_PATH)) if os.path.exists(PRIVATE_NAMES_PATH) else None
print("Name check:", "on" if NAME_PATTERNS else "OFF (private_names.txt not found)")

## 3. Pick the conversations

A fixed random sample (same `SEED` = same conversations every time), one per distinct conversation context, so near-duplicate sliding windows are not counted twice.

In [ ]:
with open(VAL_PATH, encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

unique = {}
for record in records:
    context_key = json.dumps(record["messages"][:-1], ensure_ascii=False)
    unique.setdefault(context_key, record)

rng = random.Random(SEED)
conversations = rng.sample(list(unique.values()), min(N_CONVERSATIONS, len(unique)))
SYSTEM_MESSAGE = conversations[0]["messages"][0]["content"]
real_replies = [c["messages"][-1]["content"] for c in conversations]
has_placeholder = ["[NAME]" in reply for reply in real_replies]
print(f"{len(conversations)} conversations (from {len(unique)} distinct contexts); "
      f"{sum(has_placeholder)} real replies contain [NAME]")

## 4. Helpers

- `reply_loss`: the average loss on the **final reply and its `<|im_end|>`** only, the same tokens v2 was trained on. Lower is better; `exp(loss)` is the perplexity.
- `generate_reply`: the project's decoding settings (temperature 0.4, explicit `<|im_end|>` stop).

In [ ]:
def load(path):
    model, tokenizer = FastLanguageModel.from_pretrained(model_name=path, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=False)
    model.eval()
    return model, tokenizer


@torch.no_grad()
def reply_loss(model, tokenizer, messages):
    prompt = tokenizer.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True)
    full = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(full, add_special_tokens=False)["input_ids"]
    assert full_ids[: len(prompt_ids)] == prompt_ids, "the prompt tokens changed when the reply was appended"
    im_end = tokenizer.convert_tokens_to_ids("<|im_end|>")
    reply = full_ids[len(prompt_ids):]
    reply = reply[: reply.index(im_end) + 1]
    ids = torch.tensor([prompt_ids + reply], device=model.device)
    logits = model(input_ids=ids).logits[0, len(prompt_ids) - 1 : -1].float()  # position t predicts token t+1
    return F.cross_entropy(logits, ids[0, len(prompt_ids):]).item()


@torch.no_grad()
def generate_reply(model, tokenizer, messages):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    encoded = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    eos = tokenizer.convert_tokens_to_ids("<|im_end|>")
    output = model.generate(**encoded, max_new_tokens=MAX_NEW_TOKENS, do_sample=True, temperature=0.4,
                            repetition_penalty=1.1, no_repeat_ngram_size=4, eos_token_id=eos, pad_token_id=eos)
    return tokenizer.decode(output[0, encoded["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def run_model(path):
    model, tokenizer = load(path)
    losses = [reply_loss(model, tokenizer, c["messages"]) for c in conversations]
    FastLanguageModel.for_inference(model)
    torch.manual_seed(SEED)
    replies = [generate_reply(model, tokenizer, c["messages"][:-1]) for c in conversations]
    prompt_replies = [generate_reply(model, tokenizer, [{"role": "system", "content": SYSTEM_MESSAGE},
                                                        {"role": "user", "content": p}]) for p in TEST_PROMPTS]
    del model, tokenizer  # two 3B models do not fit on a T4: release this one before loading the next
    gc.collect()
    torch.cuda.empty_cache()
    return {"losses": losses, "replies": replies, "prompt_replies": prompt_replies}

## 5. Run both models (one at a time)

In [ ]:
results = {}
for name, path in (("v1", V1_PATH), ("v2", V2_PATH)):
    print(f"Running {name} ...")
    results[name] = run_model(path)
print("Done.")

## 6. Results

**Fair subset:** the real replies that contain `[NAME]` are a disadvantage for v1, which never saw that placeholder in training. The likelihood is therefore also reported without them.

In [ ]:
def likelihood_report(indices):
    v1 = [results["v1"]["losses"][i] for i in indices]
    v2 = [results["v2"]["losses"][i] for i in indices]
    wins = sum(b < a for a, b in zip(v1, v2))
    losses = sum(b > a for a, b in zip(v1, v2))
    p = binomtest(wins, wins + losses, 0.5).pvalue if wins + losses else float("nan")
    mean_v1, mean_v2 = statistics.mean(v1), statistics.mean(v2)
    return {"n": len(indices), "v1_loss": round(mean_v1, 4), "v2_loss": round(mean_v2, 4),
            "v1_perplexity": round(2.718281828 ** mean_v1, 2), "v2_perplexity": round(2.718281828 ** mean_v2, 2),
            "v2_better_in": wins, "v1_better_in": losses, "sign_test_p": round(p, 4)}


all_idx = list(range(len(conversations)))
fair_idx = [i for i in all_idx if not has_placeholder[i]]


def count_names(replies):
    return sum(bool(find_private_names(r, NAME_PATTERNS)) for r in replies) if NAME_PATTERNS else None


report = {
    "n_conversations": len(conversations),
    "likelihood_all": likelihood_report(all_idx),
    "likelihood_fair_subset": likelihood_report(fair_idx),
    "median_reply_chars": {
        "real": statistics.median(len(r) for r in real_replies),
        "v1": statistics.median(len(r) for r in results["v1"]["replies"]),
        "v2": statistics.median(len(r) for r in results["v2"]["replies"]),
    },
    "replies_with_a_private_name": {"real": count_names(real_replies),
                                    "v1": count_names(results["v1"]["replies"]),
                                    "v2": count_names(results["v2"]["replies"])},
    "replies_with_[NAME]_placeholder": {name: sum("[NAME]" in r for r in results[name]["replies"]) for name in ("v1", "v2")},
}

print(json.dumps(report, indent=2, ensure_ascii=False))

### How to read it

- **Likelihood:** lower loss / perplexity = predicts your real replies better. `v2_better_in` counts conversations where v2 wins; a `sign_test_p` under 0.05 means the difference is unlikely to be luck. With 60 conversations, small differences will not be significant: that is what the full evaluation is for.
- **Private names:** your real replies already mention names sometimes (`real`); the goal is a model at or below that, never above. Names are matched to `private_names.txt` only, so the count is a lower bound.
- **`[NAME]` placeholder:** v2 learned from data where names were replaced, so it may write `[NAME]` literally. That is expected; at deployment an output filter can drop or rephrase it.
- **Length:** closer to `real` is better.

### Three fixed prompts, side by side

In [ ]:
for i, prompt in enumerate(TEST_PROMPTS):
    print(f"{prompt}\n  v1: {results['v1']['prompt_replies'][i]}\n  v2: {results['v2']['prompt_replies'][i]}\n")

In [ ]:
if SHOW_EXAMPLES:  # private: real conversations and replies
    for i in range(5):
        print("Contact:", conversations[i]["messages"][-2]["content"])
        print("  real:", real_replies[i])
        print("  v1:  ", results["v1"]["replies"][i])
        print("  v2:  ", results["v2"]["replies"][i], "\n")
else:
    print("Examples hidden (SHOW_EXAMPLES = False).")

## 7. Save the totals (no messages)

In [ ]:
os.makedirs(os.path.dirname(RESULTS_PATH), exist_ok=True)
with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump({"seed": SEED, **report}, f, indent=2, ensure_ascii=False)
print("Saved:", RESULTS_PATH)

## 8. Chat with v2

Change `MESSAGE` and run the cell again. The model is loaded once.

In [ ]:
MESSAGE = "عامل ايه؟"

if "chat_model" not in globals():
    chat_model, chat_tokenizer = load(V2_PATH)
    FastLanguageModel.for_inference(chat_model)

print(generate_reply(chat_model, chat_tokenizer,
                     [{"role": "system", "content": SYSTEM_MESSAGE}, {"role": "user", "content": MESSAGE}]))